# Ampelschaltung · kleine MicroPython-Vorlagen

Ein Baukasten zum **Selbstprogrammieren**, keine fertige Prüfungsabgabe. Jede Vorlage hat einen Zweck und einen kurzen Test. Die Beispiele orientieren sich an `Ampel_Code_1.1` und `Ampel_Code_1.2`, verändern diese Dateien aber nicht.

**Im Notebook:** Python-3-Kernel wählen und alle Zellen ausführen. Eine kleine Simulation ersetzt `machine.Pin` und die MicroPython-Zeitfunktionen. Keine Endlosschleifen, kein `sleep()` und keine echte Hardware nötig.

**Auf dem Board:** Den Simulationsblock durch die echten Importe ersetzen, die Pinbelegung prüfen und die benötigten Bausteine übernehmen. Die Vorlage für `while True` steht als Text bereit und wird im Notebook nicht ausgeführt.

## Bausteine

1. Hardware und Pins · 2. Zeitbedingung · 3. Flanken · 4. Zwei entprellte Taster · 5. Sensor · 6. Blinktimer · 7. LED-Muster · 8. Zustandswechsel · 9. Anforderungen speichern · 10. Sensorregeln · 11. eigenes Hauptprogramm.

**Hardware:** Board mit MicroPython, sechs LEDs (Auto: rot/gelb/grün; Fußgänger: rot/grün; Status: beliebig), sechs passende Vorwiderstände, zwei Taster, digitale Lichtschranke, Breadboard und Kabel.

## 1 · Hardware vorbereiten

Je LED: GPIO → eigener Vorwiderstand → Anode; Kathode → GND. Der Code setzt **aktiv-high** angeschlossene LEDs voraus. Taster jeweils zwischen GPIO und GND, mit `Pin.PULL_UP`: losgelassen = 1, gedrückt = 0.

Sensor: gemeinsame Masse, passende Versorgung und zulässiger Ausgangspegel. Einen 5-V-Ausgang nicht direkt mit einem 3,3-V-GPIO verbinden. Aktivpegel anhand des Sensors prüfen. Open-Drain-Ausgänge benötigen einen geeigneten Pull-up.

### Pinplan aus deinem bisherigen Code

| Signal | Beispiel-GPIO |
|---|---:|
| Auto rot / gelb / grün | 12 / 27 / 25 |
| Fußgänger rot / grün | 18 / 21 |
| Status-LED | 32 |
| Taster 1 / 2 | 4 / 5 |
| Lichtschranke | 34 |

**Das ist kein universeller Pinplan.** GPIO32/34 gibt es beispielsweise am RP2040-Pico nicht. Beim klassischen ESP32 ist GPIO34 nur Eingang und hat keinen internen Pull-up/-down. Auch andere Pins können boardspezifisch belegt sein. Vor dem Übertragen das konkrete Board-Pinout prüfen.

Auf echter Hardware statt der folgenden Simulationszelle verwenden:

```python
from machine import Pin
import time
```

Die Simulation danach auf dem Board **nicht** übernehmen.

In [1]:
# NUR NOTEBOOK: deterministische Testhardware, keine echten GPIOs.
class SimPin:
    IN, OUT, PULL_UP = 0, 1, 2
    def __init__(self, nummer, modus, pull=None, value=0):
        self.nummer, self.modus = nummer, modus
        self.wert = 1 if pull == self.PULL_UP else int(value)
    def value(self, neu=None):
        if neu is not None:
            if self.modus != self.OUT:
                raise ValueError("Eingang über set_input() verändern")
            self.wert = int(bool(neu))
        return self.wert
    def set_input(self, neu):
        if self.modus != self.IN:
            raise ValueError("Nur für Eingänge")
        self.wert = int(bool(neu))

class SimZeit:
    PERIODE = 1 << 16  # künstlich klein für Überlauftests
    def __init__(self):
        self.ms = 0
    def ticks_ms(self):
        return self.ms % self.PERIODE
    def ticks_diff(self, neu, alt):
        halb = self.PERIODE // 2
        return (neu - alt + halb) % self.PERIODE - halb
    def weiter(self, ms):
        if not 0 <= ms < self.PERIODE // 2:
            raise ValueError("Schritt außerhalb des erlaubten Testbereichs")
        self.ms += ms
        return self.ticks_ms()

Pin = SimPin
time = SimZeit()
print("Simulation aktiv – echte Pins werden nicht angesprochen.")

Simulation aktiv – echte Pins werden nicht angesprochen.


In [2]:
# VORLAGE: Nur diese Zahlen passend zu DEINEM Board ändern.
PINS = {
    "auto_rot": 12, "auto_gelb": 27, "auto_gruen": 25,
    "fuss_rot": 18, "fuss_gruen": 21, "status": 32,
    "taster1": 4, "taster2": 5, "sensor": 34,
}
# Erstellt zunächst ausgeschaltete LEDs; danach eine definierte Anzeige setzen.
auto_rot = Pin(PINS["auto_rot"], Pin.OUT, value=0)
auto_gelb = Pin(PINS["auto_gelb"], Pin.OUT, value=0)
auto_gruen = Pin(PINS["auto_gruen"], Pin.OUT, value=0)
fuss_rot = Pin(PINS["fuss_rot"], Pin.OUT, value=0)
fuss_gruen = Pin(PINS["fuss_gruen"], Pin.OUT, value=0)
status_led = Pin(PINS["status"], Pin.OUT, value=0)
taster1 = Pin(PINS["taster1"], Pin.IN, Pin.PULL_UP)
taster2 = Pin(PINS["taster2"], Pin.IN, Pin.PULL_UP)
sensor = Pin(PINS["sensor"], Pin.IN)

# Einzeltest einer LED: erst an, später separat wieder aus.
status_led.value(1)
assert status_led.value() == 1
status_led.value(0)

0

## 2 · Nicht-blockierende Zeitbedingung

**Vor der Schleife:** Startzeit speichern. **In der Schleife:** Differenz prüfen. Der Zähler hat keinen für die Aufgabe relevanten Nullpunkt. `ticks_diff(neu, alt)` berücksichtigt Überläufe, sofern der tatsächliche Abstand kleiner als die halbe Zählerperiode ist. Die echte Periode hängt von der Plattform ab.

Nach einer wiederkehrenden Aktion `start = jetzt` setzen; bei einem Zustandswechsel stattdessen den Zustandstimer zurücksetzen. Zustandstimer und Blinktimer sind unterschiedliche Variablen.

In [3]:
def zeit_abgelaufen(jetzt, start, dauer_ms):
    return time.ticks_diff(jetzt, start) >= dauer_ms

start = time.ticks_ms()
assert not zeit_abgelaufen(start, start, 1000)
jetzt = time.weiter(999)  # nur Simulation
assert not zeit_abgelaufen(jetzt, start, 1000)
jetzt = time.weiter(1)
assert zeit_abgelaufen(jetzt, start, 1000)
print("1000 ms erreicht; andere Aufgaben konnten dazwischen laufen.")

# Überlauf: von 65530 bis 14 sind 20 ms vergangen.
assert time.ticks_diff(14, 65530) == 20

1000 ms erreicht; andere Aufgaben konnten dazwischen laufen.


## 3 · Einfache Flankenerkennung

Das ist die kleine Vorlage **ohne Entprellung**. Eine fallende Flanke (`1 → 0`) bedeutet bei Pull-up „gerade gedrückt“. Der Pegel `0` allein bedeutet auch „weiterhin gehalten“ und darf keine erneute Anforderung auslösen.

In [4]:
alt = 1  # auf dem Board vor der Schleife: alt = taster1.value()
ereignisse = []
for neu in [1, 0, 0, 0, 1, 0]:  # auf dem Board pro Runde: neu = taster1.value()
    gedrueckt = alt == 1 and neu == 0
    losgelassen = alt == 0 and neu == 1
    if gedrueckt:
        ereignisse.append("Druck")
    alt = neu  # immer aktualisieren!
assert ereignisse == ["Druck", "Druck"]
print(ereignisse)

['Druck', 'Druck']


## 4 · Zwei Taster entprellen

Diese Funktion passt zum Stil deiner bisherigen Dateien. Jeder Taster braucht **eigene** Werte für Rohpegel, stabilen Pegel und Änderungszeit. Die Funktion muss für **beide** Taster in jeder Runde aufgerufen werden: nicht mit einem kurzschließenden `or` verbinden.

Ein Rohwert muss 30 ms unverändert bleiben. Erst danach liefert der Übergang zum stabilen LOW ein Ereignis. Kurze Impulse werden ignoriert. Ein beim Start bereits gedrückter Taster löst ohne erneutes Loslassen/Drücken nicht aus.

In [5]:
ENTPRELLZEIT = 30

def pruefe_taster(pin, jetzt, alt, stabil, start):
    neu = pin.value()
    druck = False
    if neu != alt:
        alt = neu
        start = jetzt
    if neu != stabil and zeit_abgelaufen(jetzt, start, ENTPRELLZEIT):
        stabil = neu
        druck = stabil == 0
    return alt, stabil, start, druck

# Vor der Hauptschleife:
roh1 = stabil1 = taster1.value()
roh2 = stabil2 = taster2.value()
zeit1 = zeit2 = time.ticks_ms()

# In jedem Schleifendurchlauf:
jetzt = time.ticks_ms()
roh1, stabil1, zeit1, druck1 = pruefe_taster(taster1, jetzt, roh1, stabil1, zeit1)
roh2, stabil2, zeit2, druck2 = pruefe_taster(taster2, jetzt, roh2, stabil2, zeit2)
neue_anforderung = druck1 or druck2
print("Neue Anforderung:", neue_anforderung)

Neue Anforderung: False


In [6]:
# TEST: Taster 1 prellt, Taster 2 wird später gedrückt.
p1, p2 = Pin(4, Pin.IN, Pin.PULL_UP), Pin(5, Pin.IN, Pin.PULL_UP)
s1, s2 = (1, 1, 0), (1, 1, 0)
gefunden = []
for ms, wert1, wert2 in [(0,1,1), (10,0,1), (12,1,1), (15,0,1),
                         (44,0,1), (45,0,1), (50,0,0), (80,0,0), (100,0,0)]:
    p1.set_input(wert1)
    p2.set_input(wert2)
    r1 = pruefe_taster(p1, ms, *s1)
    r2 = pruefe_taster(p2, ms, *s2)
    s1, s2 = r1[:3], r2[:3]
    if r1[3] or r2[3]:
        gefunden.append((ms, r1[3], r2[3]))
assert gefunden == [(45, True, False), (80, False, True)]
print("Entprellte Ereignisse:", gefunden)

Entprellte Ereignisse: [(45, True, False), (80, False, True)]


## 5 · Lichtschranke: Pegel oder Ereignis

`SENSOR_AKTIV` ist der elektrische Aktivpegel, nicht die Bedeutung des Sensors. Erst entscheiden: Erkennt die Lichtschranke ein wartendes Auto oder eine belegte Querung?

- **Pegel:** „Solange belegt, Fußgängerphase nicht beenden.“
- **Flanke:** „Bei einer neuen Erkennung genau einmal zählen oder protokollieren.“

In der späteren Steuerung nur eine zur Aufgabenstellung passende Sensorregel verwenden.

In [7]:
SENSOR_AKTIV = 1  # Bei aktiv-low auf 0 ändern.

def sensor_ist_aktiv(pin):
    return pin.value() == SENSOR_AKTIV

vorher_aktiv = sensor_ist_aktiv(sensor)  # vor der Schleife
# In der Schleife:
aktiv = sensor_ist_aktiv(sensor)
neue_erkennung = aktiv and not vorher_aktiv
vorher_aktiv = aktiv
print("Aktiv:", aktiv, "Neue Erkennung:", neue_erkennung)

# NUR TEST:
sensor.set_input(SENSOR_AKTIV)
assert sensor_ist_aktiv(sensor)
sensor.set_input(1 - SENSOR_AKTIV)
assert not sensor_ist_aktiv(sensor)

Aktiv: False Neue Erkennung: False


## 6 · Eine LED blinken lassen

Diese Funktion schaltet höchstens einmal pro Aufruf um. Sie holt verpasste Intervalle nicht nach. Ein Umschaltintervall von 250 ms ergibt bei regelmäßigen Aufrufen eine vollständige An/Aus-Periode von 500 ms, also 2 Hz.

Beim Eintritt in einen Blinkzustand `blink_start = jetzt` und `blink_an = True` setzen. Zwei unabhängige Blinkmuster benötigen getrennte Zeitstempel.

In [8]:
def blink_schritt(jetzt, start, an, intervall_ms):
    if zeit_abgelaufen(jetzt, start, intervall_ms):
        return jetzt, not an
    return start, an

blink_start, blink_an = 0, True
for jetzt in [0, 249, 250, 499, 500]:
    blink_start, blink_an = blink_schritt(jetzt, blink_start, blink_an, 250)
    print(jetzt, "ms →", int(blink_an))
assert blink_an is True

# Zwei unabhängige Timer (beispielsweise Fuß- und Status-LED):
fuss_start, fuss_an = 0, True
status_start, status_an = 0, False
jetzt = 250
fuss_start, fuss_an = blink_schritt(jetzt, fuss_start, fuss_an, 250)
status_start, status_an = blink_schritt(jetzt, status_start, status_an, 500)
assert (fuss_an, status_an) == (False, False)

0 ms → 1
249 ms → 1
250 ms → 0
499 ms → 0
500 ms → 1


## 7 · LED-Muster zentral setzen

Die Zustandslogik entscheidet, **welche Phase** aktiv ist; diese Funktion legt die dazugehörigen LED-Werte fest. Reihenfolge: Auto rot, gelb, grün; Fußgänger rot, grün. Die sechste LED zeigt eine gespeicherte Anforderung.

Die Tabelle ist ein Beispiel: Rot-Gelb vor Auto-Grün, keine gleichzeitige Freigabe von Auto und Fußgängern. Es handelt sich um einen Lernaufbau, nicht um eine Steuerung für echten Straßenverkehr.

In [9]:
AUTO_GRUEN = "AUTO_GRUEN"
AUTO_GELB = "AUTO_GELB"
ALLE_ROT_VOR = "ALLE_ROT_VOR"
FUSS_GRUEN = "FUSS_GRUEN"
FUSS_BLINKT = "FUSS_BLINKT"
ALLE_ROT_NACH = "ALLE_ROT_NACH"
AUTO_ROT_GELB = "AUTO_ROT_GELB"

MUSTER = {
    AUTO_GRUEN:    (0, 0, 1, 1, 0),
    AUTO_GELB:     (0, 1, 0, 1, 0),
    ALLE_ROT_VOR:  (1, 0, 0, 1, 0),
    FUSS_GRUEN:    (1, 0, 0, 0, 1),
    ALLE_ROT_NACH: (1, 0, 0, 1, 0),
    AUTO_ROT_GELB: (1, 1, 0, 1, 0),
}

def ampelwerte(zustand, blink_an=True):
    if zustand == FUSS_BLINKT:
        return (1, 0, 0, 0, int(blink_an))
    return MUSTER.get(zustand, (1, 0, 0, 1, 0))  # unbekannt → beide rot

def ausgaenge_setzen(zustand, blink_an, anforderung):
    pins = (auto_rot, auto_gelb, auto_gruen, fuss_rot, fuss_gruen)
    werte = ampelwerte(zustand, blink_an)
    # Zuerst alle aus, dann das neue Muster: keine alte Grünfreigabe bleibt stehen.
    for pin in pins:
        pin.value(0)
    for pin, wert in zip(pins, werte):
        pin.value(wert)
    status_led.value(int(anforderung))

# Definierter Start für den Lernaufbau:
ausgaenge_setzen(ALLE_ROT_VOR, True, False)
assert auto_rot.value() == fuss_rot.value() == 1
for zustand in list(MUSTER) + [FUSS_BLINKT, "UNBEKANNT"]:
    for blink_an in (False, True):
        werte = ampelwerte(zustand, blink_an)
        assert not (werte[2] and werte[4]), "Beide Grünfreigaben gleichzeitig!"
print("Alle LED-Muster geprüft.")

Alle LED-Muster geprüft.


## 8 · Zustandswechsel und Eintrittsaktionen

Kleine Vorlage: neuer Zustand, neuer Zustandstimer und zurückgesetzter Blinktimer. Hier werden Werte zurückgegeben, damit du kein `global` brauchst.

Die Konstanten sind **Beispielzeiten zum Üben**, keine vorgegebenen Prüfungsanforderungen. Deine beiden bisherigen Dateien enthalten zusätzliche Phasen und unterschiedliche Sensorregeln; übernimm nur, was die jeweilige Aufgabe verlangt.

In [10]:
AUTO_MIN = 3000
GELB_ZEIT = 2000
ROT_ZEIT = 1000
FUSS_MIN = 5000
BLINK_DAUER = 2000
BLINK_INTERVALL = 250

def wechsel_zustand(neu, jetzt):
    return neu, jetzt, jetzt, True

zustand, zustand_start, blink_start, blink_an = wechsel_zustand(AUTO_GRUEN, 0)
# Beispiel: nach einer Anforderung und abgelaufener Mindestzeit.
jetzt = 3000
zustand, zustand_start, blink_start, blink_an = wechsel_zustand(AUTO_GELB, jetzt)
assert zustand_start == blink_start == 3000
assert blink_an is True
print("Wechsel nach", zustand)

Wechsel nach AUTO_GELB


### Übergangstabelle als Planungs-Vorlage

| Von | Bedingung (Beispiel) | Nach |
|---|---|---|
| AUTO_GRUEN | Anforderung und Mindestgrünzeit abgelaufen | AUTO_GELB |
| AUTO_GELB | Gelbzeit abgelaufen | ALLE_ROT_VOR |
| ALLE_ROT_VOR | Räumzeit abgelaufen | FUSS_GRUEN |
| FUSS_GRUEN | Mindestzeit und passende Sensorregel | FUSS_BLINKT |
| FUSS_BLINKT | Blinkdauer abgelaufen; ggf. Sensorvorrang | ALLE_ROT_NACH |
| ALLE_ROT_NACH | Räumzeit, ggf. Sensorbedingung erfüllt | AUTO_ROT_GELB |
| AUTO_ROT_GELB | Rot-Gelb-Zeit abgelaufen | AUTO_GRUEN |

Sensorpriorität explizit festlegen. Beispiel für eine belegte Fußgängerquerung: In FUSS_BLINKT zuerst `belegt` prüfen und nach FUSS_GRUEN zurückkehren, erst danach das Ende der Blinkdauer prüfen.

In einer `if`/`elif`-Kette wird pro Durchlauf nur der Zweig des aktuellen Zustands ausgeführt. So verhindern wir mehrere aufeinanderfolgende Zustandswechsel innerhalb derselben Runde.

In [11]:
# VORLAGE: zwei einzelne Zustandszweige – um weitere elif-Zweige ergänzen.
def zwei_zustandszweige(zustand, jetzt, zustand_start, anforderung):
    neu = zustand
    if zustand == AUTO_GRUEN:
        if anforderung and zeit_abgelaufen(jetzt, zustand_start, AUTO_MIN):
            neu = AUTO_GELB
            anforderung = False  # Anforderung wird jetzt bedient.
    elif zustand == AUTO_GELB:
        if zeit_abgelaufen(jetzt, zustand_start, GELB_ZEIT):
            neu = ALLE_ROT_VOR
    return neu, anforderung

assert zwei_zustandszweige(AUTO_GRUEN, 2999, 0, True) == (AUTO_GRUEN, True)
assert zwei_zustandszweige(AUTO_GRUEN, 3000, 0, True) == (AUTO_GELB, False)
assert zwei_zustandszweige(AUTO_GELB, 5000, 3000, False)[0] == ALLE_ROT_VOR

## 9 · Eine Tasteranforderung speichern

Ein Tastendruck ist nur in einer Runde `True`; die Anforderung muss bis zur Bedienung gespeichert bleiben. Im Beispiel werden neue Anforderungen nur während AUTO_GRUEN akzeptiert. Während anderer Phasen werden sie ignoriert. Falls deine Aufgabe Warteschlangen verlangt, diese Regel gezielt ändern.

In [12]:
def anforderung_speichern(anforderung, zustand, druck1, druck2):
    if zustand == AUTO_GRUEN and (druck1 or druck2):
        anforderung = True
    return anforderung

anforderung = False
anforderung = anforderung_speichern(anforderung, AUTO_GRUEN, False, True)
assert anforderung
anforderung = anforderung_speichern(anforderung, AUTO_GRUEN, False, False)
assert anforderung  # bleibt gespeichert, obwohl niemand mehr drückt
assert not anforderung_speichern(False, FUSS_GRUEN, True, False)
print("Anforderung bleibt bis zur Bedienung erhalten.")

Anforderung bleibt bis zur Bedienung erhalten.


## 10 · Sensorregel wählen: zwei unterschiedliche Aufgaben

### Variante A: Lichtschranke überwacht die Fußgängerquerung

Eine belegte Querung verhindert das Beenden der Fußgänger-Grünphase. Erst wenn der Sensor mindestens eine Sekunde durchgehend frei war und die Mindestgrünzeit vorbei ist, darf die nächste Phase beginnen. Belegung setzt den Freizeittimer zurück.

`None` bedeutet „keine laufende Freimessung“. Nicht `if frei_start:` schreiben: Der gültige Zeitstempel 0 wäre sonst fälschlich unwahr.

In [13]:
FREI_ZEIT = 1000

def pruefe_freizeit(belegt, jetzt, frei_start):
    if belegt:
        frei_start = None
    elif frei_start is None:
        frei_start = jetzt
    frei = (frei_start is not None
            and zeit_abgelaufen(jetzt, frei_start, FREI_ZEIT))
    return frei_start, frei

frei_start = None
frei_start, frei = pruefe_freizeit(False, 0, frei_start)
assert frei_start == 0 and not frei
frei_start, frei = pruefe_freizeit(False, 1000, frei_start)
assert frei
frei_start, frei = pruefe_freizeit(True, 1001, frei_start)
assert frei_start is None and not frei
# Verwendung in FUSS_GRUEN:
jetzt, zustand_start = 6000, 0
sensor_frei = False
darf_blinken = zeit_abgelaufen(jetzt, zustand_start, FUSS_MIN) and sensor_frei
assert not darf_blinken

### Variante B: Lichtschranke erkennt ein wartendes Auto

Andere Bedeutung, andere Logik: Wartet ein Auto, endet die Fußgängerphase frühestens nach der Mindestzeit; ohne Auto spätestens nach einer Maximalzeit. Diese Regel überwacht **keine** belegte Fußgängerquerung. Sie ist eine Alternative zu Variante A, kein zusätzlicher Freigabebefehl.

In [14]:
FUSS_MAX = 10000

def fussphase_beenden(jetzt, start, auto_erkannt):
    dauer = time.ticks_diff(jetzt, start)
    return (dauer >= FUSS_MIN and auto_erkannt) or dauer >= FUSS_MAX

assert not fussphase_beenden(4999, 0, True)
assert fussphase_beenden(5000, 0, True)
assert not fussphase_beenden(5000, 0, False)
assert fussphase_beenden(10000, 0, False)
print("Mindest- und Maximalzeit geprüft.")

Mindest- und Maximalzeit geprüft.


## 11 · Eigenes Hauptprogramm zusammensetzen

Die folgende Vorlage ist bewusst **unvollständig**: Sie verdrahtet die Bausteine, enthält jedoch nur die zwei bereits gezeigten Zustandszweige. Es fehlt der restliche Zyklus aus der Übergangstabelle. Ohne Ergänzung bleibt das Programm in ALLE_ROT_VOR. Der Text wird nicht automatisch auf dem Board ausgeführt.

**Reihenfolge:** Zeit lesen → beide Taster prüfen → Sensor lesen → Anforderung merken → genau einen Zustandszweig auswerten → Eintrittsaktionen → Blinken → LEDs setzen.

Auf dem Board zuerst die echten Importe, die geprüfte Hardwarekonfiguration, Konstanten und benötigten Funktionen aus den vorigen Abschnitten übernehmen. Keine Testzellen und keine simulierten `weiter()`-/`set_input()`-Aufrufe übernehmen.

In [15]:
hauptschleife_vorlage = """# ERGÄNZEN: echte Importe, Hardware und Bausteinfunktionen voranstellen.
zustand_start = time.ticks_ms()
zustand = AUTO_GRUEN
blink_start, blink_an = zustand_start, True
anforderung = False
frei_start = None
roh1 = stabil1 = taster1.value()
roh2 = stabil2 = taster2.value()
zeit1 = zeit2 = zustand_start

ausgaenge_setzen(zustand, blink_an, anforderung)
while True:
    jetzt = time.ticks_ms()
    roh1, stabil1, zeit1, druck1 = pruefe_taster(taster1, jetzt, roh1, stabil1, zeit1)
    roh2, stabil2, zeit2, druck2 = pruefe_taster(taster2, jetzt, roh2, stabil2, zeit2)
    aktiv = sensor_ist_aktiv(sensor)
    anforderung = anforderung_speichern(anforderung, zustand, druck1, druck2)

    # TODO: Bedeutung des Sensors festlegen und passende Regel ergänzen.
    # Variante A, zum Beispiel:
    # frei_start, sensor_frei = pruefe_freizeit(aktiv, jetzt, frei_start)

    # TODO: diese Funktion um die übrigen Zustandszweige erweitern.
    neu, anforderung = zwei_zustandszweige(zustand, jetzt, zustand_start, anforderung)
    if neu != zustand:
        zustand, zustand_start, blink_start, blink_an = wechsel_zustand(neu, jetzt)

    if zustand == FUSS_BLINKT:
        blink_start, blink_an = blink_schritt(jetzt, blink_start, blink_an, BLINK_INTERVALL)
    ausgaenge_setzen(zustand, blink_an, anforderung)
"""
print(hauptschleife_vorlage)

# ERGÄNZEN: echte Importe, Hardware und Bausteinfunktionen voranstellen.
zustand_start = time.ticks_ms()
zustand = AUTO_GRUEN
blink_start, blink_an = zustand_start, True
anforderung = False
frei_start = None
roh1 = stabil1 = taster1.value()
roh2 = stabil2 = taster2.value()
zeit1 = zeit2 = zustand_start

ausgaenge_setzen(zustand, blink_an, anforderung)
while True:
    jetzt = time.ticks_ms()
    roh1, stabil1, zeit1, druck1 = pruefe_taster(taster1, jetzt, roh1, stabil1, zeit1)
    roh2, stabil2, zeit2, druck2 = pruefe_taster(taster2, jetzt, roh2, stabil2, zeit2)
    aktiv = sensor_ist_aktiv(sensor)
    anforderung = anforderung_speichern(anforderung, zustand, druck1, druck2)

    # TODO: Bedeutung des Sensors festlegen und passende Regel ergänzen.
    # Variante A, zum Beispiel:
    # frei_start, sensor_frei = pruefe_freizeit(aktiv, jetzt, frei_start)

    # TODO: diese Funktion um die übrigen Zustandszweige erweitern.
    neu, anforderung = zwei_zustandszweige(zustand, jetzt, zustand_s

## 12 · Kleine Aufgaben zum Ergänzen

1. Ergänze `AUTO_GELB → ALLE_ROT_VOR → FUSS_GRUEN` mit eigenen Zeitbedingungen.
2. Wähle **eine** Sensorbedeutung. Begründe den Übergang aus FUSS_GRUEN.
3. Ergänze FUSS_BLINKT und die Rückkehr bis AUTO_GRUEN.
4. Variante A: Was passiert, wenn die Querung genau beim Ende der Blinkdauer belegt wird? Formuliere und implementiere den Vorrang.
5. Zusatz: Lass die Status-LED beim Warten unabhängig blinken. Verwende einen eigenen Timer.
6. Falls die Aufgabe Auto-Grün-Blinken fordert: ergänze dafür einen eigenen Zustand und setze beim Eintritt den Blinktimer zurück.

### Tests, die du selbst durchführen solltest

| Versuch | Erwartung |
|---|---|
| Taster lange halten | genau eine Anforderung |
| Beide Taster gleichzeitig drücken | eine gespeicherte Anforderung |
| Kurze Rohwertwechsel unter 30 ms | kein zusätzlicher Druck |
| Drücken während einer ignorierten Phase | keine spätere versteckte Anforderung |
| Sensor während Blinkphase verändern | Reaktion im nächsten Durchlauf nach deiner Prioritätsregel |
| Zeitbedingung genau an der Grenze | Übergang mit `>=` |
| Neue Blinkphase starten | definierter LED-Anfangswert und neuer Timer |
| Sensor beim Einschalten aktiv | bewusst festgelegte Startreaktion |
| Vollständiger Zyklus | nie Auto-Grün und Fuß-Grün gleichzeitig |

### Merksätze

- Jede Aufgabe mit eigenem Zeitverlauf braucht ihren eigenen Zeitstempel.
- Flanke = Ereignis; Pegel = Zustand; Entprellung = Stabilitätsprüfung.
- Timer beim Zustandswechsel neu initialisieren.
- Keine dauernden `print()`-Ausgaben in der schnellen Schleife.
- Erst Übergangstabelle, dann Code. Halte Prioritäten und die Behandlung neuer Anforderungen ausdrücklich fest.

**Validierung dieses Notebooks:** Alle ausführbaren Beispiele sind für Python-Simulation ausgelegt; Hardware und vollständiger Ampelzyklus müssen nach deiner Ergänzung gesondert geprüft werden.

Referenzen: [machine.Pin](https://docs.micropython.org/en/latest/library/machine.Pin.html) · [ticks_ms / ticks_diff](https://docs.micropython.org/en/latest/library/time.html)